# Extracción de embeddings VGG-16 — TrashNet completo V2

Etapa 2 del análisis. Compara cada imagen original con su versión transformada en cuatro transformaciones, tres intensidades y cinco bloques de VGG-16. Produce 151 620 registros (2527 × 12 × 5).

El procesamiento se guarda por condición. Si Colab se desconecta, vuelve a ejecutar el cuaderno: las condiciones completas se omiten automáticamente.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 1. Configuración y reproducibilidad

In [ ]:
import json, platform, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as torch_F
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASE_DIR = Path('/content/gdrive/MyDrive/2026-0')
ORIGINAL_ROOT = BASE_DIR / 'DatasetOriginalCompleto'
BALANCED_822_ROOT = BASE_DIR / 'DatasetOriginal-137-Por-Clase'
TRANSFORMED_ROOT = BASE_DIR / 'TrashNet_transformaciones_completo_png'
RESULTS_ROOT = BASE_DIR / 'Resultados_VGG16_TrashNet_completo_v2_resize224'
CONDITION_ROOT = RESULTS_ROOT / 'condiciones'
FINAL_CSV = RESULTS_ROOT / 'embeddings_metricas_VGG16_TrashNet_completo_v2.csv'
METADATA_JSON = RESULTS_ROOT / 'entorno_ejecucion.json'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CONDITION_ROOT.mkdir(parents=True, exist_ok=True)

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
EXPECTED_COUNTS = {'cardboard': 403, 'glass': 501, 'metal': 410, 'paper': 594, 'plastic': 482, 'trash': 137}
CONDITIONS = {
    'brightness': {'low': 1.2, 'medium': 1.4, 'high': 1.6},
    'contrast': {'low': 1.2, 'medium': 1.4, 'high': 1.6},
    'rotation': {'low': 10, 'medium': 30, 'high': 60},
    'zoom': {'low': 1.10, 'medium': 1.25, 'high': 1.50},
}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
BATCH_SIZE = 32
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', DEVICE)
print('Resultados:', RESULTS_ROOT)

## 2. Inventario y correspondencia original–transformada

In [ ]:
def list_images(folder):
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p: p.name.lower()
    )

if not ORIGINAL_ROOT.exists():
    raise FileNotFoundError(f'No existe {ORIGINAL_ROOT}')
if not BALANCED_822_ROOT.exists():
    raise FileNotFoundError(f'No existe {BALANCED_822_ROOT}; se necesita para identificar exactamente las 822 imágenes antiguas.')
if not TRANSFORMED_ROOT.exists():
    raise FileNotFoundError(f'No existe {TRANSFORMED_ROOT}. Ejecuta primero el cuaderno de transformaciones.')

originals_by_class = {}
for class_name in CLASSES:
    paths = list_images(ORIGINAL_ROOT / class_name)
    originals_by_class[class_name] = paths
    observed = len(paths)
    expected = EXPECTED_COUNTS[class_name]
    print(f'{class_name:10s}: {observed:4d}/{expected:4d}')
    if observed != expected:
        raise ValueError(f'Conteo incorrecto para {class_name}: {observed}, esperado {expected}')

balanced_names_by_class = {}
for class_name in CLASSES:
    balanced_paths = list_images(BALANCED_822_ROOT / class_name)
    balanced_names_by_class[class_name] = {p.name for p in balanced_paths}
    if len(balanced_paths) != 137:
        raise ValueError(f'El subconjunto antiguo de {class_name} tiene {len(balanced_paths)} imágenes; se esperaban 137.')
    complete_names = {p.name for p in originals_by_class[class_name]}
    missing_from_complete = balanced_names_by_class[class_name] - complete_names
    if missing_from_complete:
        raise ValueError(f'Hay imágenes del subconjunto 822 que no aparecen en el completo para {class_name}: {sorted(missing_from_complete)[:10]}')

TOTAL_IMAGES = sum(len(v) for v in originals_by_class.values())
print('Total:', TOTAL_IMAGES)
assert TOTAL_IMAGES == 2527

## 3. VGG-16 congelada y captura de cinco bloques

In [ ]:
weights = VGG16_Weights.IMAGENET1K_V1
# Debe coincidir exactamente con el flujo JPG anterior.
# No usar weights.transforms(), porque añade Resize(256)+CenterCrop(224).
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
model = vgg16(weights=weights).to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

LAYER_INDEX = {
    'Block1': 3,   # Conv1_2 + ReLU
    'Block2': 8,   # Conv2_2 + ReLU
    'Block3': 15,  # Conv3_3 + ReLU
    'Block4': 22,  # Conv4_3 + ReLU
    'Block5': 29,  # Conv5_3 + ReLU
}
activations = {}
handles = []
for block_name, layer_index in LAYER_INDEX.items():
    def save_activation(module, inputs, output, name=block_name):
        activations[name] = output
    handles.append(model.features[layer_index].register_forward_hook(save_activation))

print('Capas capturadas:')
for name, idx in LAYER_INDEX.items():
    print(name, idx, model.features[idx])

## 4. Dataset pareado y métricas

In [ ]:
class PairedTrashNetDataset(Dataset):
    def __init__(self, original_root, transformed_root, class_names, transformation, level):
        self.items = []
        for class_name in class_names:
            for original_path in originals_by_class[class_name]:
                transformed_path = transformed_root / f'{transformation}_{level}' / class_name / f'{original_path.stem}.png'
                if not transformed_path.exists():
                    raise FileNotFoundError(f'Falta la transformación: {transformed_path}')
                self.items.append((class_name, original_path, transformed_path))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        class_name, original_path, transformed_path = self.items[index]
        with Image.open(original_path) as image:
            original = preprocess(image.convert('RGB'))
        with Image.open(transformed_path) as image:
            transformed = preprocess(image.convert('RGB'))
        return original, transformed, class_name, original_path.name

def embedding_metrics(original_embedding, transformed_embedding, eps=1e-8):
    euclidean = torch.linalg.vector_norm(original_embedding - transformed_embedding, dim=1)
    cosine = torch_F.cosine_similarity(original_embedding, transformed_embedding, dim=1, eps=eps)
    relative_euclidean = torch.linalg.vector_norm(original_embedding - transformed_embedding, dim=1) / (
        torch.linalg.vector_norm(original_embedding, dim=1) + eps
    )
    return euclidean, cosine, relative_euclidean

## 5. Extracción por condición
Cada lote concatena originales y transformadas, realiza una sola inferencia y luego separa las dos mitades. No se guarda el embedding completo: solo las métricas necesarias para el artículo.

In [ ]:
def extract_condition(transformation, level, value):
    output_csv = CONDITION_ROOT / f'{transformation}_{level}.csv'
    expected_rows = TOTAL_IMAGES * len(LAYER_INDEX)

    if output_csv.exists():
        previous = pd.read_csv(output_csv)
        if len(previous) == expected_rows:
            print(f'OMITIDA {transformation}-{level}: ya está completa ({expected_rows} filas).')
            return output_csv
        print(f'Recalculando {output_csv.name}: contiene {len(previous)} de {expected_rows} filas.')

    dataset = PairedTrashNetDataset(ORIGINAL_ROOT, TRANSFORMED_ROOT, CLASSES, transformation, level)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(), persistent_workers=(NUM_WORKERS > 0)
    )
    rows = []

    with torch.inference_mode():
        for original, transformed, class_names, filenames in tqdm(loader, desc=f'{transformation}-{level}'):
            original = original.to(DEVICE, non_blocking=True)
            transformed = transformed.to(DEVICE, non_blocking=True)
            batch_size = original.shape[0]
            combined = torch.cat([original, transformed], dim=0)
            activations.clear()
            _ = model(combined)

            for block_name in LAYER_INDEX:
                pooled = activations[block_name].mean(dim=(2, 3))
                original_embedding = pooled[:batch_size]
                transformed_embedding = pooled[batch_size:]
                euclidean, cosine, relative_euclidean = embedding_metrics(original_embedding, transformed_embedding)

                euclidean = euclidean.cpu().numpy()
                cosine = cosine.cpu().numpy()
                relative_euclidean = relative_euclidean.cpu().numpy()
                for i in range(batch_size):
                    rows.append({
                        'image_id': Path(filenames[i]).stem,
                        'original_filename': filenames[i],
                        'in_balanced_822': filenames[i] in balanced_names_by_class[class_names[i]],
                        'original_format': Path(filenames[i]).suffix.lower().lstrip('.'),
                        'transformed_format': 'png',
                        'class': class_names[i],
                        'transformation': transformation,
                        'level': level,
                        'value': value,
                        'block': block_name,
                        'euclidean_distance': float(euclidean[i]),
                        'cosine_similarity': float(cosine[i]),
                        'relative_euclidean_distance': float(relative_euclidean[i]),
                    })

    result = pd.DataFrame(rows)
    if len(result) != expected_rows:
        raise RuntimeError(f'{transformation}-{level}: {len(result)} filas; se esperaban {expected_rows}.')
    if result[['euclidean_distance', 'cosine_similarity', 'relative_euclidean_distance']].isna().any().any():
        raise RuntimeError(f'{transformation}-{level}: se detectaron valores faltantes.')
    result.to_csv(output_csv, index=False, float_format='%.10f')
    print(f'GUARDADA {output_csv.name}: {len(result)} filas.')
    return output_csv

In [ ]:
condition_files = []
for transformation, levels in CONDITIONS.items():
    for level, value in levels.items():
        condition_files.append(extract_condition(transformation, level, value))

for handle in handles:
    handle.remove()
print('Extracción terminada.')

## 6. Unión, auditoría y exportación final

In [ ]:
condition_files = [CONDITION_ROOT / f'{t}_{level}.csv' for t, levels in CONDITIONS.items() for level in levels]
missing = [str(p) for p in condition_files if not p.exists()]
if missing:
    raise FileNotFoundError(f'Faltan archivos de condición: {missing}')

results = pd.concat([pd.read_csv(p) for p in condition_files], ignore_index=True)
expected_total = TOTAL_IMAGES * 12 * len(LAYER_INDEX)
key = ['class', 'original_filename', 'transformation', 'level', 'block']

print('Filas:', len(results), '| esperadas:', expected_total)
print('Duplicados por clave:', results.duplicated(key).sum())
print('Valores faltantes:', int(results.isna().sum().sum()))
print('Filas pertenecientes al subconjunto 822:', int(results.in_balanced_822.sum()), '| esperadas:', 822 * 12 * len(LAYER_INDEX))
print('Rango coseno:', results.cosine_similarity.min(), results.cosine_similarity.max())
print('Rango distancia relativa:', results.relative_euclidean_distance.min(), results.relative_euclidean_distance.max())

assert len(results) == expected_total
assert results.duplicated(key).sum() == 0
assert not results.isna().any().any()
assert int(results.in_balanced_822.sum()) == 822 * 12 * len(LAYER_INDEX)
assert results.cosine_similarity.between(-1.000001, 1.000001).all()
assert (results.relative_euclidean_distance >= 0).all()

results.to_csv(FINAL_CSV, index=False, float_format='%.10f')

environment = {
    'seed': SEED,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'device': str(DEVICE),
    'weights': 'VGG16_Weights.IMAGENET1K_V1',
    'preprocessing': str(preprocess),
    'preprocessing_note': 'Direct Resize((224,224)) to match the previous JPG experiment; no center crop.',
    'layer_indices': LAYER_INDEX,
    'total_original_images': TOTAL_IMAGES,
    'total_result_rows': len(results),
}
METADATA_JSON.write_text(json.dumps(environment, indent=2), encoding='utf-8')
print('CSV final:', FINAL_CSV)
print('Metadatos:', METADATA_JSON)

## 7. Comprobación descriptiva inicial

In [ ]:
level_order = {'low': 0, 'medium': 1, 'high': 2}
block_order = {f'Block{i}': i for i in range(1, 6)}
summary = (
    results.groupby(['transformation', 'level', 'value', 'block'], as_index=False)
    .agg(
        n=('cosine_similarity', 'size'),
        cosine_mean=('cosine_similarity', 'mean'),
        cosine_sd=('cosine_similarity', 'std'),
        relative_euclidean_mean=('relative_euclidean_distance', 'mean'),
        relative_euclidean_sd=('relative_euclidean_distance', 'std'),
    )
)
summary['_level'] = summary.level.map(level_order)
summary['_block'] = summary.block.map(block_order)
summary = summary.sort_values(['transformation', '_level', '_block']).drop(columns=['_level', '_block'])
summary_path = RESULTS_ROOT / 'resumen_descriptivo_micro.csv'
summary.to_csv(summary_path, index=False, float_format='%.10f')
display(summary)
print('Resumen:', summary_path)

## Resultado esperado
Al finalizar, comparte `embeddings_metricas_VGG16_TrashNet_completo.csv`. Con ese archivo se construirán las tablas micro/macro, las pruebas de Friedman y Kendall's W, los post hoc con Holm, los análisis por clase y la comparación formal con los 822 casos.